# Лабораторна робота — Тестування в Python

---
# Частина 1: Загальне тестування та unittest

## 1.1 Перевірка даних через `assert` та `raise ValueError`

In [ ]:
# assert — зупиняє програму якщо умова хибна
def check_positive_assert(number: float) -> float:
    assert number > 0, "Число має бути більшим за нуль!"
    return number

# raise ValueError — повідомляє про неправильний аргумент
def create_name(name: str) -> str:
    if not name:
        raise ValueError("Ім'я не може бути порожнім")
    return name

print("=== Правильне введення ===")
print(check_positive_assert(5))
print(create_name("Іван"))

print("\n=== Неправильне введення ===")
try:
    check_positive_assert(-1)
except AssertionError as e:
    print(f"AssertionError: {e}")

try:
    create_name("")
except ValueError as e:
    print(f"ValueError: {e}")

## 1.2 Функція підрахунку голосних літер

In [ ]:
def count_vowels(text: str) -> int:
    """Рахує кількість голосних літер у рядку."""
    vowels = set("aeiouAEIOUаеєиіїоуюяАЕЄИІЇОУЮЯ")
    return sum(1 for ch in text if ch in vowels)

print(f'apple     -> {count_vowels("apple")}')    # 2
print(f'gym       -> {count_vowels("gym")}')      # 0
print(f'(порожній)-> {count_vowels("")}')         # 0
print(f'12345     -> {count_vowels("12345")}')    # 0
print(f'привіт    -> {count_vowels("привіт")}')   # 2
print(f'україна   -> {count_vowels("україна")}')  # 4

## 1.3 Клас Figure

In [ ]:
class Figure:
    VALID_TYPES = ("квадрат", "прямокутник", "трикутник")

    def __init__(self, figure_type: str, length: float):
        if length <= 0:
            raise ValueError("Довжина має бути більшою за нуль")
        if figure_type not in self.VALID_TYPES:
            raise ValueError(f"Невідомий тип фігури: {figure_type}")
        self._figure_type = figure_type
        self._length = length

    @property
    def get_figure_type(self) -> str:
        return self._figure_type

    @property
    def get_figure_length(self) -> float:
        return self._length

    @property
    def get_angles(self) -> int:
        angles = {"квадрат": 4, "прямокутник": 4, "трикутник": 3}
        return angles[self._figure_type]

    def __str__(self) -> str:
        return f"{self._figure_type} зі стороною {self._length}"

# Перевірка вручну
for name, length in [("квадрат", 5), ("прямокутник", 3), ("трикутник", 7)]:
    f = Figure(name, length)
    print(f"Тип: {f.get_figure_type}, Довжина: {f.get_figure_length}, Кути: {f.get_angles}")

## 1.4 Тести з unittest (setUp, tearDown, subTest, mock)

In [ ]:
import unittest
from unittest.mock import patch

def ask_user_number() -> str:
    return input("Введіть число: ")


class TestValidation(unittest.TestCase):
    def test_assert_positive_ok(self):
        self.assertEqual(5, check_positive_assert(5))
    def test_assert_negative_raises(self):
        with self.assertRaises(AssertionError):
            check_positive_assert(-1)
    def test_assert_zero_raises(self):
        with self.assertRaises(AssertionError):
            check_positive_assert(0)
    def test_create_name_ok(self):
        self.assertEqual("Іван", create_name("Іван"))
    def test_create_name_empty_raises(self):
        with self.assertRaises(ValueError):
            create_name("")


class TestCountVowels(unittest.TestCase):
    def test_english_vowels(self):
        self.assertEqual(2, count_vowels("apple"))
    def test_no_vowels(self):
        self.assertEqual(0, count_vowels("gym"))
    def test_all_vowels(self):
        self.assertEqual(3, count_vowels("aei"))
    def test_empty_string(self):
        self.assertEqual(0, count_vowels(""))
    def test_digits_only(self):
        self.assertEqual(0, count_vowels("12345"))
    def test_ukrainian_letters(self):
        self.assertEqual(2, count_vowels("привіт"))
        self.assertEqual(4, count_vowels("україна"))


class TestFigureType(unittest.TestCase):
    def test_square_type(self):
        self.assertEqual("квадрат", Figure("квадрат", 5).get_figure_type)
    def test_rectangle_type(self):
        self.assertEqual("прямокутник", Figure("прямокутник", 3).get_figure_type)
    def test_triangle_type(self):
        self.assertEqual("трикутник", Figure("трикутник", 7).get_figure_type)


class TestFigureLength(unittest.TestCase):
    def setUp(self):
        self.square = Figure("квадрат", 10)
    def tearDown(self):
        del self.square
    def test_length_is_correct(self):
        self.assertEqual(10, self.square.get_figure_length)
    def test_zero_length_raises(self):
        with self.assertRaises(ValueError):
            Figure("квадрат", 0)
    def test_negative_length_raises(self):
        with self.assertRaises(ValueError):
            Figure("квадрат", -5)
    def test_unknown_type_raises(self):
        with self.assertRaises(ValueError):
            Figure("коло", 5)


class TestFigureAngles(unittest.TestCase):
    def test_square_angles(self):
        self.assertEqual(4, Figure("квадрат", 5).get_angles)
    def test_rectangle_angles(self):
        self.assertEqual(4, Figure("прямокутник", 5).get_angles)
    def test_triangle_angles(self):
        self.assertEqual(3, Figure("трикутник", 5).get_angles)


class TestFigureSubTest(unittest.TestCase):
    def test_figure_lengths_subtest(self):
        for length in [1, 5, 10]:
            with self.subTest(length=length):
                self.assertEqual(length, Figure("квадрат", length).get_figure_length)
    def test_all_figure_types_subtest(self):
        for figure_type in ["квадрат", "прямокутник", "трикутник"]:
            with self.subTest(figure_type=figure_type):
                self.assertEqual(figure_type, Figure(figure_type, 4).get_figure_type)


class TestMockInput(unittest.TestCase):
    @patch("builtins.input", return_value="5")
    def test_ask_user_number_returns_value(self, mock_input):
        result = ask_user_number()
        self.assertEqual("5", result)
        mock_input.assert_called_once()
    @patch("builtins.input", return_value="")
    def test_ask_user_number_empty(self, mock_input):
        self.assertEqual("", ask_user_number())


unittest.main(argv=[""], exit=False, verbosity=2)

---
# Частина 2: Тестування з бібліотекою pytest

In [ ]:
# Встановлення pytest
import subprocess
subprocess.run(["pip", "install", "pytest", "-q"], capture_output=True)

In [ ]:
import pytest

# --- Базові тести у стилі pytest (прості функції без класу) ---

def test_triangle_type():
    triangle = Figure("трикутник", 4)
    assert triangle.get_figure_type == "трикутник"

def test_square_length():
    square = Figure("квадрат", 5)
    assert square.get_figure_length == 5

def test_invalid_length():
    with pytest.raises(ValueError):
        Figure("квадрат", 0)

def test_invalid_type():
    with pytest.raises(ValueError):
        Figure("коло", 5)

print("Функції тестів визначено успішно")

In [ ]:
# --- Параметризація ---

@pytest.mark.parametrize("figure_type", Figure.VALID_TYPES)
def test_allowed_figure(figure_type):
    assert Figure(figure_type, 1).get_figure_type == figure_type

@pytest.mark.parametrize("length,expected", [
    (1, 1),
    (5, 5),
    (10, 10),
])
def test_figure_length_param(length, expected):
    assert Figure("квадрат", length).get_figure_length == expected

print("Параметризовані тести визначено")

In [ ]:
# --- Fixtures ---

@pytest.fixture
def square():
    return Figure("квадрат", 5)

@pytest.fixture(scope="module")
def allowed_figures():
    return Figure.VALID_TYPES

def test_square_type_fixture(square):
    assert square.get_figure_type == "квадрат"

def test_square_length_fixture(square):
    assert square.get_figure_length == 5

print("Fixtures визначено")

In [ ]:
# --- Запуск pytest у notebook через ipytest ---
import subprocess
result = subprocess.run(
    ["python", "-m", "pytest", "--tb=short", "-v", "--no-header", "-p", "no:cacheprovider"],
    capture_output=True, text=True, cwd="."
)
print(result.stdout)
print(result.stderr)

### Різниця між unittest.TestCase та функцією pytest

| | unittest | pytest |
|---|---|---|
| Синтаксис | Клас наслідує TestCase | Звичайна функція |
| Перевірка | self.assertEqual(a, b) | assert a == b |
| Винятки | self.assertRaises(Error) | pytest.raises(Error) |
| Параметризація | subTest | @pytest.mark.parametrize |
| Підготовка даних | setUp/tearDown | @pytest.fixture |

---
# Частина 3: Покриття коду (coverage)

In [ ]:
# Встановлення coverage
import subprocess
subprocess.run(["pip", "install", "coverage", "pytest-cov", "-q"], capture_output=True)
print("coverage встановлено")

In [ ]:
# Демонстрація покриття вручну через coverage.py
import coverage

cov = coverage.Coverage()
cov.start()

# Запускаємо тести
f1 = Figure("квадрат", 5)
_ = f1.get_figure_type
_ = f1.get_figure_length
_ = f1.get_angles

f2 = Figure("прямокутник", 3)
_ = f2.get_angles

f3 = Figure("трикутник", 7)
_ = f3.get_angles

try:
    Figure("квадрат", 0)
except ValueError:
    pass

try:
    Figure("коло", 5)
except ValueError:
    pass

cov.stop()
cov.save()

print("Покриття зібрано успішно")

In [ ]:
# Звіт покриття
import subprocess

result = subprocess.run(
    ["python", "-m", "coverage", "report", "-m"],
    capture_output=True, text=True
)
print(result.stdout if result.stdout else "Запустіть спочатку: coverage run -m pytest")

# Команди для запуску у терміналі:
print("\n=== Команди для терміналу ===")
print("coverage run -m pytest")
print("coverage report -m")
print("coverage html  # HTML звіт у папці htmlcov/")
print("pytest --cov=. --cov-report=term-missing -v")

### Різниця між покриттям рядків і розгалужень

- **Покриття рядків (line coverage)** — показує які рядки коду були виконані
- **Покриття розгалужень (branch coverage)** — показує які умови `if/else` були перевірені в обох напрямках

Наприклад якщо є `if length <= 0` і тест перевіряє тільки правильне значення — рядок виконано, але гілка `True` не покрита.

---
# Частина 4: Інтеграційне тестування (Flask + SQLite)

In [ ]:
# Встановлення Flask
import subprocess
subprocess.run(["pip", "install", "flask", "-q"], capture_output=True)
print("Flask встановлено")

In [ ]:
# app.py — Flask застосунок з SQLite
import sqlite3
import json
import os
from flask import Flask, request, jsonify

app = Flask(__name__)

DB_PATH = "tasks.db"

def init_db(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL
        )
    """)
    conn.commit()
    conn.close()

def export_tasks(filepath, db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    rows = conn.execute("SELECT id, title FROM tasks").fetchall()
    conn.close()
    tasks = [{"id": r[0], "title": r[1]} for r in rows]
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(tasks, f, ensure_ascii=False)
    return tasks

@app.route("/tasks", methods=["GET"])
def get_tasks():
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("SELECT id, title FROM tasks").fetchall()
    conn.close()
    return jsonify([{"id": r[0], "title": r[1]} for r in rows])

@app.route("/tasks", methods=["POST"])
def create_task():
    data = request.get_json()
    if not data or "title" not in data:
        return jsonify({"error": "title is required"}), 400
    conn = sqlite3.connect(DB_PATH)
    conn.execute("INSERT INTO tasks (title) VALUES (?)", (data["title"],))
    conn.commit()
    conn.close()
    return jsonify({"message": "created"}), 201

@app.route("/tasks/<int:task_id>", methods=["DELETE"])
def delete_task(task_id):
    conn = sqlite3.connect(DB_PATH)
    conn.execute("DELETE FROM tasks WHERE id = ?", (task_id,))
    conn.commit()
    conn.close()
    return jsonify({"message": "deleted"}), 200

print("Flask застосунок визначено")

In [ ]:
# Інтеграційні тести
import tempfile
import os

# --- Тест файлової системи ---
def test_export_tasks():
    with tempfile.TemporaryDirectory() as tmp:
        db_path = os.path.join(tmp, "test.db")
        json_path = os.path.join(tmp, "tasks.json")

        init_db(db_path)
        conn = sqlite3.connect(db_path)
        conn.execute("INSERT INTO tasks (title) VALUES (?)", ("Тестове завдання",))
        conn.commit()
        conn.close()

        tasks = export_tasks(json_path, db_path)

        assert os.path.exists(json_path)
        with open(json_path, encoding="utf-8") as f:
            data = json.load(f)
        assert len(data) == 1
        assert data[0]["title"] == "Тестове завдання"
    print("test_export_tasks: OK")

# --- Тест порожнього списку ---
def test_export_empty_tasks():
    with tempfile.TemporaryDirectory() as tmp:
        db_path = os.path.join(tmp, "empty.db")
        json_path = os.path.join(tmp, "empty.json")
        init_db(db_path)
        tasks = export_tasks(json_path, db_path)
        assert tasks == []
    print("test_export_empty_tasks: OK")

# --- Тест бази даних ---
def test_db_two_tasks_order():
    with tempfile.TemporaryDirectory() as tmp:
        db_path = os.path.join(tmp, "order.db")
        init_db(db_path)
        conn = sqlite3.connect(db_path)
        conn.execute("INSERT INTO tasks (title) VALUES (?)", ("Перше",))
        conn.execute("INSERT INTO tasks (title) VALUES (?)", ("Друге",))
        conn.commit()
        rows = conn.execute("SELECT title FROM tasks ORDER BY id").fetchall()
        conn.close()
        assert rows[0][0] == "Перше"
        assert rows[1][0] == "Друге"
    print("test_db_two_tasks_order: OK")

# --- Тести Flask клієнта ---
def test_create_task_http():
    init_db()
    client = app.test_client()
    response = client.post("/tasks", json={"title": "Вивчити pytest"})
    assert response.status_code == 201
    print("test_create_task_http: OK")

def test_get_tasks_http():
    client = app.test_client()
    response = client.get("/tasks")
    assert response.status_code == 200
    print("test_get_tasks_http: OK")

def test_create_task_without_title():
    client = app.test_client()
    response = client.post("/tasks", json={})
    assert response.status_code == 400
    print("test_create_task_without_title: OK")

def test_delete_task_http():
    init_db()
    conn = sqlite3.connect(DB_PATH)
    conn.execute("INSERT INTO tasks (title) VALUES (?)", ("Для видалення",))
    conn.commit()
    task_id = conn.execute("SELECT last_insert_rowid()").fetchone()[0]
    conn.close()

    client = app.test_client()
    response = client.delete(f"/tasks/{task_id}")
    assert response.status_code == 200
    print("test_delete_task_http: OK")

# Запускаємо всі інтеграційні тести
print("=== Інтеграційні тести ===")
test_export_tasks()
test_export_empty_tasks()
test_db_two_tasks_order()
test_create_task_http()
test_get_tasks_http()
test_create_task_without_title()
test_delete_task_http()
print("\nВсі інтеграційні тести пройшли успішно!")

### Які компоненти взаємодіють у тестах

| Тест | Компоненти |
|------|------------|
| test_export_tasks | SQLite + файлова система (JSON) |
| test_db_two_tasks_order | SQLite |
| test_create_task_http | HTTP (Flask) + SQLite |
| test_delete_task_http | HTTP (Flask) + SQLite |

**Юніт-тест** перевіряє одну функцію ізольовано.  
**Інтеграційний тест** перевіряє спільну роботу кількох компонентів.  
**Системний тест** перевіряє програму так як її використовує користувач.

---
# Висновок

- **Що зроблено:** реалізовано 4 частини лабораторної роботи з тестування в Python
- **Мета досягнута:** написано юніт-тести (unittest), тести з pytest, зібрано покриття коду та виконано інтеграційне тестування Flask-застосунку
- **Нові знання:** `assert`, `raise ValueError`, `setUp/tearDown`, `subTest`, `mock patch`, `pytest.fixture`, `parametrize`, `coverage`, інтеграційне тестування з Flask та SQLite
- **Всі завдання виконано успішно**
- **Складність:** налаштування середовища на Windows (PATH для pip та git, Python 3.15 та сумісність з pyyaml)
- **Формат здачі через GitHub зручний** — дозволяє бачити історію змін
- **Побажання:** додати більше прикладів з реальних проєктів